# DiD-BCF — B1_selection_obs (linearity_degree=2)

**Workstream B1 · canonical DiD (selection on unobservables)**

continuity check: selection on observables only

Fits DiD-BCF on the `B1_selection_obs` scenario at `linearity_degree=2` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.7 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_selection_obs",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_selection_obs_lin_2] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_selection_obs: 100%|██████████| 100/100 [4:29:51<00:00, 161.92s/fit]

[B1_selection_obs_lin_2] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_selection_obs_lin_2.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_selection_obs,2,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.278667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
1,canonical,B1_selection_obs,2,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.358667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
2,canonical,B1_selection_obs,2,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.324667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
3,canonical,B1_selection_obs,2,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.401333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
4,canonical,B1_selection_obs,2,200,0,ES,k=0,NaN,NaN,0.0,...,0.278667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_selection_obs,2,200,ATT,ATT,power,-0.598096,-3.933413,0.34,...,0.964240,1.343110,0.716598,3.933413,1.0,0.0,0.879998,3.935127,0.375241,5.048295
1,canonical,B1_selection_obs,2,200,ES,k=0,power,-0.588274,-3.926920,0.38,...,1.217241,1.208673,0.721035,3.926920,1.0,0.0,0.892217,3.928027,0.453036,31.745743
2,canonical,B1_selection_obs,2,200,ES,k=1,power,-0.567057,-3.931930,0.43,...,1.190606,1.396951,0.693953,3.931930,1.0,0.0,0.850594,3.933832,0.473805,4.615785
3,canonical,B1_selection_obs,2,200,ES,k=2,power,-0.630635,-3.939844,0.42,...,1.190810,1.520813,0.738048,3.939844,1.0,0.0,0.900330,3.942068,0.461099,4.012075
4,canonical,B1_selection_obs,2,200,ES,k=3,power,-0.606420,-3.934958,0.41,...,1.255755,1.635765,0.768352,3.934958,1.0,0.0,0.942791,3.937709,0.441917,3.765171
5,canonical,B1_selection_obs,2,200,GATT,g=4_t=4,power,-0.588274,-3.926920,0.38,...,1.217241,1.208673,0.721035,3.926920,1.0,0.0,0.892217,3.928027,0.453036,31.745743
6,canonical,B1_selection_obs,2,200,GATT,g=4_t=5,power,-0.567057,-3.931930,0.43,...,1.190606,1.396951,0.693953,3.931930,1.0,0.0,0.850594,3.933832,0.473805,4.615785
7,canonical,B1_selection_obs,2,200,GATT,g=4_t=6,power,-0.630635,-3.939844,0.42,...,1.190810,1.520813,0.738048,3.939844,1.0,0.0,0.900330,3.942068,0.461099,4.012075
8,canonical,B1_selection_obs,2,200,GATT,g=4_t=7,power,-0.606420,-3.934958,0.41,...,1.255755,1.635765,0.768352,3.934958,1.0,0.0,0.942791,3.937709,0.441917,3.765171


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_selection_obs,2,200,corrected,100,404.32,1.191142,0.339581,1.015473,...,0.256402,0.065952,0.280403,0.111967,0.335706,0.127305,0.992200,0.349105,1.213603,0.450988
1,canonical,B1_selection_obs,2,200,plain,100,404.32,4.025204,0.111928,3.933413,...,1.000304,0.016911,0.000000,0.000000,0.000000,0.000000,1.150221,0.059113,1.446549,0.072676
